In [11]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.utils import check_random_state

In [12]:
from brain_connectome_reservoir import ConnectomeReservoir as reservoir0
from reservoir import Reservoir as reservoir1
from reservoir import Reservoir2 as reservoir2
from reservoir import Reservoir3 as reservoir3

variants = [reservoir0, reservoir1, reservoir2, reservoir3]

In [13]:
def one_hot(y, n_classes):
    Y = np.zeros((y.shape[0], n_classes), dtype=float)
    Y[np.arange(y.shape[0]), y] = 1.0
    return Y

def add_bias(X):
    return np.hstack([X, np.ones((X.shape[0], 1))])

def solve_ridge(X, Y, alpha):
    I = np.eye(X.shape[1])
    return np.linalg.solve(X.T @ X + alpha * I, X.T @ Y)

def collect_states(reservoir, U, n_classes=3):
    try:
        out = reservoir.forward(U, collect_states=True)
    except TypeError:
        out = reservoir.forward(U)

    if isinstance(out, tuple):
        cand = [a for a in out if isinstance(a, np.ndarray) and a.ndim == 2 and a.shape[0] == U.shape[0]]
    elif isinstance(out, np.ndarray) and out.ndim == 2 and out.shape[0] == U.shape[0]:
        cand = [out]
    else:
        raise RuntimeError("Could not infer states from reservoir.forward(...) output.")

    if not cand:
        raise RuntimeError("No 2D candidate with matching time dimension returned by forward().")


    nN = getattr(reservoir, "n_neurons", None)
    if nN is not None:
        for a in cand:
            if a.shape[1] == nN:
                return a


    cand.sort(key=lambda a: a.shape[1], reverse=True)
    return cand[0]


In [14]:
X, y = load_iris(return_X_y=True)
n_classes = len(np.unique(y))

In [15]:

RNG_SEED   = 42
ALPHA      = 1e-2
GRAPH_DIR  = "C://Users/ricca/OneDrive/università/BAINSA/ESN/dati/network_architecture/folder_path_83"


Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RNG_SEED
)

scaler = StandardScaler()
Xtr = scaler.fit_transform(Xtr)
Xte = scaler.transform(Xte)

Ytr = one_hot(ytr, n_classes)
Yte = one_hot(yte, n_classes)



for i, reservoir in enumerate(variants):
    
    if i==0:
        common_kwargs = dict(
            n_inputs = X.shape[1],
            folder   = GRAPH_DIR,      # or graph_dir=GRAPH_DIR / path=GRAPH_DIR
            rhow     = 1.25,
            leak_range = (0.1, 0.3),
            seed     = RNG_SEED,
            symmetric = True,
        )
        res_train = reservoir(target_size=200, **common_kwargs)
        res_test  = reservoir(target_size=200, **common_kwargs)
    else:
        common_kwargs = dict(
            n_inputs = X.shape[1],
            rhow     = 1.25,
            leak_range = (0.1, 0.3),
        )
        res_train = reservoir(n_neurons=200, **common_kwargs)
        res_test  = reservoir(n_neurons=200, **common_kwargs)

    Utr = Xtr.astype(float)  # shape (T_train, 4)
    Ute = Xte.astype(float)  # shape (T_test,  4)
    
    X_states_tr = collect_states(res_train, Utr, n_classes=n_classes)  # (T_train, N)
    X_states_te = collect_states(res_test,  Ute, n_classes=n_classes)  # (T_test,  N)

    Xb_tr = add_bias(X_states_tr)           # add bias column
    Wout  = solve_ridge(Xb_tr, Ytr, ALPHA)  # (N+1, C)


    Yhat_tr = Xb_tr @ Wout
    mse_tr  = mean_squared_error(Ytr, Yhat_tr)
    acc_tr  = accuracy_score(ytr, np.argmax(Yhat_tr, axis=1))


    Xb_te  = add_bias(X_states_te)
    Yhat_te = Xb_te @ Wout
    mse_te  = mean_squared_error(Yte, Yhat_te)
    acc_te  = accuracy_score(yte, np.argmax(Yhat_te, axis=1))

    print(f"Reservoir + Ridge Readout" + str(i))
    print(f"  Train MSE: {mse_tr:.6f} | Train Acc: {acc_tr:.3f}")
    print(f"  Test  MSE: {mse_te:.6f} | Test  Acc: {acc_te:.3f}")

    Xb_lin_tr = add_bias(Xtr)
    W_lin     = solve_ridge(Xb_lin_tr, Ytr, ALPHA)
    Yhat_lin_tr = add_bias(Xtr) @ W_lin
    Yhat_lin_te = add_bias(Xte) @ W_lin

    mse_lin_tr = mean_squared_error(Ytr, Yhat_lin_tr)
    mse_lin_te = mean_squared_error(Yte, Yhat_lin_te)
    acc_lin_tr = accuracy_score(ytr, np.argmax(Yhat_lin_tr, axis=1))
    acc_lin_te = accuracy_score(yte, np.argmax(Yhat_lin_te, axis=1))

    print(f"\nBaseline (no reservoir) — Ridge on standardized features"  + str(i))
    print(f"  Train MSE: {mse_lin_tr:.6f} | Train Acc: {acc_lin_tr:.3f}")
    print(f"  Test  MSE: {mse_lin_te:.6f} | Test  Acc: {acc_lin_te:.3f}")


Reservoir + Ridge Readout0
  Train MSE: 0.036670 | Train Acc: 0.981
  Test  MSE: 0.232597 | Test  Acc: 0.800

Baseline (no reservoir) — Ridge on standardized features0
  Train MSE: 0.083529 | Train Acc: 0.895
  Test  MSE: 0.105206 | Test  Acc: 0.733
Reservoir + Ridge Readout1
  Train MSE: 0.012417 | Train Acc: 1.000
  Test  MSE: 31.996440 | Test  Acc: 0.422

Baseline (no reservoir) — Ridge on standardized features1
  Train MSE: 0.083529 | Train Acc: 0.895
  Test  MSE: 0.105206 | Test  Acc: 0.733
Reservoir + Ridge Readout2
  Train MSE: 0.194351 | Train Acc: 0.495
  Test  MSE: 0.186112 | Test  Acc: 0.533

Baseline (no reservoir) — Ridge on standardized features2
  Train MSE: 0.083529 | Train Acc: 0.895
  Test  MSE: 0.105206 | Test  Acc: 0.733
Reservoir + Ridge Readout3
  Train MSE: 0.014288 | Train Acc: 0.990
  Test  MSE: 22.299794 | Test  Acc: 0.333

Baseline (no reservoir) — Ridge on standardized features3
  Train MSE: 0.083529 | Train Acc: 0.895
  Test  MSE: 0.105206 | Test  Acc: 0.73